# 🦈 Shark Attacks — Exploratory Data Analysis
**Ironhack Data Analytics Bootcamp | Week 2 Project**

---

### Project Overview
This notebook contains the **main workflow and demonstration** of the Shark Attacks project.  
It covers:
1. Dataset import and initial exploration  
2. Data cleaning (executed via functions from `.py` files)  
3. Exploratory Data Analysis (EDA) — statistics and visualizations  
4. Hypothesis validation and insights

> **Data source:** [Global Shark Attack File — sharkattackfile.net](https://www.sharkattackfile.net/incidentlog.htm)

---
## 📋 About the Dataset

The Global Shark Attack File (GSAF) logs shark incidents worldwide. Entries are categorized as follows:

| Category | Description |
|---|---|
| **Unprovoked** | Shark initiated contact without human provocation |
| **Provoked** | Human drew "first blood" (spearing, hooking, capturing) |
| **Watercraft** | Boat bitten or rammed; hooked/netted cases are classed as provoked |
| **Air/Sea Disaster** | Incidents during maritime or aviation accidents |
| **Questionable** | Insufficient data to confirm shark involvement |

> *All individuals survived unless noted otherwise.*

---
## 1️⃣ Setup — Imports & Configuration

In [3]:
import pandas as pd
#import numpy as np

print('✅ Libraries loaded successfully')

✅ Libraries loaded successfully


---
## 2️⃣ Import Dataset

The file `GSAF5.xls` is stored in the **same folder** as this notebook, so we use a relative path.  
This ensures the notebook works on any computer that clones the project repository.

In [2]:
# Relative path — works for anyone who clones this repo
file_location = "GSAF5.xls"

shark_df = pd.read_excel(file_location)

print(f'✅ Dataset loaded: {shark_df.shape[0]:,} rows × {shark_df.shape[1]} columns')

✅ Dataset loaded: 7,087 rows × 23 columns


---
## 3️⃣ Initial Data Exploration

Before cleaning, we inspect the raw dataset to understand its structure, column types, and completeness.

In [4]:
# First 5 rows — quick snapshot of the data
shark_df.head()

,Date,Year,Type,Country,State,Location,Activity,Name,Sex,Age,Injury,Fatal Y/N,Time,Species,Source,pdf,href formula,href,Case Number,Case Number.1,original order,Unnamed: 21,Unnamed: 22
0,14/04,2026.0,UNprovoked,Maldives,Gaafu Alif Atoll,Kooddoo,Swimming,Not stated - on honeymoon,M,?,Leg strpped off flesh later amputated in hospital,N,?,Unknown,The U.S. Sun: Simon De Marchi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3rd April,2026.0,Unprovoked,Australia,South Australia,Middleton Beach Fleurieu Peninsula Adelaide,Surfing,Oliver Tokic-Bensley,M,16,Bite wound to R ankle,N,?,Bronze Whaler,ABC News: The Guardian: Andrew Currie and Bob...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,26th March,2026.0,Unprovoked,Bahamas,Andros Island,Fresh Creek,Swimming,Australian woman,F,22,Severe injuries to R arm,N,1730hrs,Unknown,WIC News: Melissa Michaelson,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,25th March,2026.0,Unprovoked,Australia,South Australia,Cape Jaffa Limestone Coast,Diving,Luke Kuhn,M,?,NaN,N,?,Great White Shark 3.5m (11.5ft),Bob Myatt GSAF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,22nd-23rd March,2026.0,Unprovoked,Australia,NSW,Little Avalon Beach,Surfing,Unknown,M,30+,Graze injuries R leg and ankle,N,?,Unknown,Melissa Michaelson: Instagram,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# Data types and non-null counts per column
shark_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7087 entries, 0 to 7086
Data columns (total 23 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Date            7087 non-null   object 
 1   Year            7085 non-null   float64
 2   Type            7069 non-null   object 
 3   Country         7037 non-null   object 
 4   State           6600 non-null   object 
 5   Location        6520 non-null   object 
 6   Activity        6504 non-null   object 
 7   Name            6869 non-null   object 
 8   Sex             6509 non-null   object 
 9   Age             4093 non-null   object 
 10  Injury          7051 non-null   object 
 11  Fatal Y/N       6526 non-null   object 
 12  Time            3560 non-null   object 
 13  Species         3956 non-null   object 
 14  Source          7067 non-null   object 
 15  pdf             6799 non-null   object 
 16  href formula    6794 non-null   object 
 17  href            6796 non-null   o

In [6]:
# Descriptive statistics for categorical columns
shark_df.describe(include='object')

,Date,Type,Country,State,Location,Activity,Name,Sex,Age,Injury,Fatal Y/N,Time,Species,Source,pdf,href formula,href,Case Number,Case Number.1,Unnamed: 21,Unnamed: 22
count,7087,7069,7037,6600,6520,6504,6869,6509,4093,7051,6526,3560,3956,7067,6799,6794,6796,6798,6797,1,2
unique,6126,14,253,948,4631,1612,5801,11,252,4197,12,477,1746,5413,6789,6784,6776,6777,6775,1,2
top,1957,Unprovoked,USA,Florida,"New Smyrna Beach, Volusia County",Surfing,male,M,16,FATAL,N,Afternoon,White shark,"K. McMurray, TrackingSharks.com",1907.10.16.R-HongKong.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,2021.07.23,2012.09.02.b,stopped here,Teramo
freq,9,5237,2580,1192,191,1151,679,5682,92,863,4943,215,194,131,2,2,4,2,2,1,1


### 🔍 Issues Identified

| Column | Issue | Recommended Action |
|---|---|---|
| `Date` | Multiple formats (strings), inconsistent | Standardize to DD.MM.YYYY or extract Year only |
| `Year` | Float type instead of integer | Convert to `int` |
| `Age` | Mixed formats (strings, `?`, `30+`) | Standardize to integer; unknown → NaN |
| `Type` | Typos creating extra categories (e.g. `UNprovoked`) | Standardize to 5 official categories |
| `Sex` | 11 unique values — should be 2 (M/F) | Clean and standardize |
| `Country`, `State`, `Activity` | Typos creating duplicate entries | Normalize text (strip, title case, deduplicate) |
| `Time` | Very sparse, mixed formats | Categorize into time-of-day buckets or drop |
| `pdf`, `href formula`, `href` | Metadata columns, not needed for analysis | Drop |
| `Case Number.1` | Likely duplicate of `Case Number` | Verify and drop if redundant |
| `original order` | Unclear purpose | Verify and drop if redundant |
| `Unnamed: 21`, `Unnamed: 22` | Only 1–2 non-null values | Drop |

> **Next step:** All cleaning logic is implemented in `cleaning_functions.py` and called in Section 4.

---
## 4️⃣ Data Cleaning

Cleaning functions are stored in a separate `.py` file to keep this notebook clean and modular.  
We import and call the main cleaning function here.

In [8]:
# Import cleaning functions from the .py file
# from cleaning_functions import clean_shark_df   # ← uncomment when .py file is ready

# Placeholder — cleaning steps will be added here
shark_clean = shark_df.copy()
print('⚠️  Cleaning functions not yet applied — using raw copy for now')

⚠️  Cleaning functions not yet applied — using raw copy for now


In [9]:
# Drop columns with no analytical value
cols_to_drop = ['pdf', 'href formula', 'href', 'Case Number.1', 
                'original order', 'Unnamed: 21', 'Unnamed: 22']

shark_clean = shark_clean.drop(columns=cols_to_drop, errors='ignore')

print(f'✅ Columns after dropping: {shark_clean.shape[1]}')
print(shark_clean.columns.tolist())

✅ Columns after dropping: 16
['Date', 'Year', 'Type', 'Country', 'State', 'Location', 'Activity', 'Name', 'Sex', 'Age', 'Injury', 'Fatal Y/N', 'Time', 'Species ', 'Source', 'Case Number']


In [10]:
# Standardize the 'Type' column — fix typos and normalize capitalization
type_mapping = {
    'UNprovoked': 'Unprovoked',
    'unprovoked': 'Unprovoked',
    'UNPROVOKED': 'Unprovoked',
    'provoked': 'Provoked',
    'PROVOKED': 'Provoked',
    'Boatomg': 'Watercraft',
    'Sea Disaster': 'Sea Disaster',
}

shark_clean['Type'] = shark_clean['Type'].replace(type_mapping)

print('Type value counts after cleaning:')
print(shark_clean['Type'].value_counts())

Type value counts after cleaning:
Type
Unprovoked             5239
Provoked                642
Invalid                 552
Watercraft              355
Sea Disaster            242
Questionable             26
Boat                      7
 Provoked                 2
?                         1
Unconfirmed               1
Unverified                1
Under investigation       1
Name: count, dtype: int64


In [11]:
# Standardize 'Sex' column — keep only M and F
print('Original Sex values:')
print(shark_clean['Sex'].value_counts(dropna=False))

shark_clean['Sex'] = shark_clean['Sex'].str.strip().str.upper()
shark_clean.loc[~shark_clean['Sex'].isin(['M', 'F']), 'Sex'] = np.nan

print('\nCleaned Sex values:')
print(shark_clean['Sex'].value_counts(dropna=False))

Original Sex values:
Sex
M        5682
F         811
NaN       578
M           5
?           2
F           2
N           2
 M          1
m           1
lli         1
M x 2       1
.           1
Name: count, dtype: int64

Cleaned Sex values:
Sex
M      5689
F       813
NaN     585
Name: count, dtype: int64


---
## 5️⃣ Exploratory Data Analysis (EDA)

With the cleaned dataset, we now explore distributions and patterns to form hypotheses.

In [ ]:
# Distribution of incident types


In [ ]:
# Top 10 activities associated with shark attacks


In [ ]:
# Top 10 countries by number of incidents


In [ ]:
# Incidents over time (Year) — filter to modern era for clarity

---
## 6️⃣ Hypothesis Validation

We test the following hypotheses using aggregation and filtering:

### 🔬 Hypothesis 1: "Fill in hypothesis #1"

In [ ]:
#simple code to test hypotesis

### 🔬 Hypothesis 2: "Male victims are significantly more common than female victims"

In [ ]:
#simple code to test hypotesisout()

---
## 7️⃣ Aggregation & Pivot Tables

In [ ]:
# Pivot: Top countries × Incident type


---
## 8️⃣ Key Insights & Conclusions

*(To be completed after hypothesis testing)*

1. **Conclusion 1:** fill fill fill .....
2. **Geography:** The contry X dominates incident counts, driven by X location's high activity.
3. **Gender:** X victims are significantly overrepresented, likely reflecting greater participation in high-risk water activities.
4. **Trend:** Recorded incidents have (increased/decreased) over time — likely due to .....
5. **Data quality:** Heavy cleaning required, especially for `X`, `X`, and `Sex` columns.

---
*Notebook by Diana Carolina Yule Burbano & Irene Fafian — Ironhack Data Analytics Bootcamp, Week 2*